In [11]:
!pip install streamlit pyngrok -q

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import os

BRISC_PATH = "/content/drive/MyDrive/BRISC_ML_Project"

print(os.listdir(BRISC_PATH))

['features', 'results', 'notebooks', 'graphs', 'classification_task']


In [14]:
!pip install streamlit pandas plotly -q

In [15]:
%%writefile /content/app.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# =========================
# PAGE CONFIG
# =========================
st.set_page_config(
    page_title="BRISC 2025 | Brain Tumor Classification",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="expanded"
)

# =========================
# DESIGN TOKENS
# =========================
# Light canvas + a dark "signal" sidebar for contrast, glow accents throughout.
BG = "#F5F8FC"
PANEL = "#FFFFFF"
PANEL_ALT = "#FBFDFF"
BORDER = "rgba(23, 35, 60, 0.09)"
BORDER_STRONG = "rgba(15, 165, 150, 0.55)"
TEXT = "#17233C"
TEXT_SUB = "#4B5875"
TEXT_MUTED = "#8592AC"
TEAL = "#0FA596"      # Glioma / primary signal
INDIGO = "#5B72E0"    # Meningioma
GREEN = "#10B981"     # No Tumor
AMBER = "#F59E0B"     # Pituitary

SIDEBAR_BG = "#0E1526"
SIDEBAR_BORDER = "rgba(255, 255, 255, 0.08)"
SIDEBAR_TEXT = "#E8EDF7"
SIDEBAR_MUTED = "#8D98B3"

CLASS_COLORS = {
    "Glioma": TEAL,
    "Meningioma": INDIGO,
    "No Tumor": GREEN,
    "Pituitary": AMBER,
}

MODEL_COLOR_SEQUENCE = [TEAL, INDIGO, AMBER, "#D6558E", GREEN]

# =========================
# CUSTOM CSS
# =========================
st.markdown(f"""
<style>

html, body, [class*="css"] {{
    font-family: 'Inter', 'Segoe UI', 'Helvetica Neue', Arial, sans-serif;
}}

/* Never touch icon fonts — forcing a text font onto them turns glyphs into
   literal ligature names like "keyboard_double_arrow_left". Streamlit's
   icons render via the "Material Symbols Rounded" font, so re-assert that
   directly rather than trying to revert (revert rolls back past Streamlit's
   own icon font rule too, which is why that approach didn't work). */
[data-testid*="Icon"],
[data-testid*="Icon"] *,
[data-testid*="Collapse"] [data-testid*="Icon"],
button[kind="header"] span,
[data-testid="collapsedControl"] span {{
    font-family: "Material Symbols Rounded", "Material Icons" !important;
}}

html, body {{
    background-color: {BG} !important;
}}

.stApp,
[data-testid="stAppViewContainer"],
[data-testid="stMain"],
.main, .block-container {{
    background: {BG} !important;
    background-image:
        radial-gradient(ellipse 900px 500px at 12% -10%, rgba(15,165,150,0.10), transparent 60%),
        radial-gradient(ellipse 800px 550px at 100% 0%, rgba(91,114,224,0.08), transparent 55%) !important;
    color: {TEXT};
}}

[data-testid="stHeader"] {{
    background: transparent !important;
}}

section[data-testid="stSidebar"] {{
    background-color: {SIDEBAR_BG} !important;
    border-right: 1px solid {SIDEBAR_BORDER};
}}

section[data-testid="stSidebar"] * {{
    color: {SIDEBAR_TEXT} !important;
    font-family: 'Inter', sans-serif;
}}

section[data-testid="stSidebar"] hr {{
    border-color: {SIDEBAR_BORDER} !important;
}}

section[data-testid="stSidebar"] [data-testid="stCaptionContainer"] * {{
    color: {SIDEBAR_MUTED} !important;
}}

section[data-testid="stSidebar"] label:hover {{
    color: {TEAL} !important;
}}

/* ---------- Navigation ---------- */

section[data-testid="stSidebar"] [data-testid="stRadio"] > div {{
    gap: 6px !important;
}}

section[data-testid="stSidebar"] [data-testid="stRadio"] label {{
    border-radius: 10px !important;
    padding: 10px 12px !important;
    margin: 2px 0 !important;
    transition: all 0.2s ease !important;
    border: 1px solid transparent !important;
}}

section[data-testid="stSidebar"] [data-testid="stRadio"] label:hover {{
    background: rgba(15, 165, 150, 0.10) !important;
    border-color: rgba(15, 165, 150, 0.20) !important;
}}

section[data-testid="stSidebar"] [data-testid="stRadio"] label[data-checked="true"] {{
    background: linear-gradient(
        90deg,
        rgba(15, 165, 150, 0.18),
        rgba(91, 114, 224, 0.10)
    ) !important;
    border-color: rgba(15, 165, 150, 0.35) !important;
}}

section[data-testid="stSidebar"] [data-testid="stRadio"] label p {{
    font-size: 13.5px !important;
    font-weight: 500 !important;
    margin: 0 !important;
}}

.brand-mark {{
    display: flex;
    align-items: center;
    gap: 10px;
    padding: 4px 0 2px 0;
}}

.brand-dot {{
    width: 10px;
    height: 10px;
    border-radius: 50%;
    background: {TEAL};
    box-shadow: 0 0 10px 2px {TEAL};
    flex-shrink: 0;
}}

.brand-name {{
    font-family: 'Space Grotesk', sans-serif;
    font-weight: 700;
    font-size: 19px;
    color: {TEXT};
    letter-spacing: 0.2px;
}}

.brand-sub {{
    color: {TEXT_MUTED};
    font-size: 12.5px;
    margin: 2px 0 14px 20px;
    line-height: 1.5;
}}

hr, .sidebar-rule {{
    border-color: {BORDER} !important;
}}

/* ---------- Typography ---------- */

.main-title {{
    font-family: 'Space Grotesk', sans-serif;
    font-size: 36px;
    font-weight: 700;
    color: {TEXT};
    margin-bottom: 6px;
    letter-spacing: -0.3px;
}}

.subtitle {{
    font-size: 15.5px;
    color: {TEXT_SUB};
    margin-bottom: 32px;
    max-width: 640px;
    line-height: 1.6;
}}

.section-header {{
    display: flex;
    align-items: center;
    gap: 12px;
    margin-top: 34px;
    margin-bottom: 16px;
}}

.section-bar {{
    width: 4px;
    height: 22px;
    border-radius: 3px;
    background: linear-gradient(180deg, {TEAL}, {INDIGO});
    box-shadow: 0 0 8px rgba(79,209,197,0.5);
}}

.section-title {{
    font-family: 'Space Grotesk', sans-serif;
    font-size: 21px;
    font-weight: 600;
    color: {TEXT};
}}

/* ---------- Cards ---------- */

.card {{
    background: {PANEL};
    padding: 26px 28px;
    border-radius: 16px;
    border: 1px solid {BORDER};
    margin-bottom: 20px;
    box-shadow: 0 8px 24px -16px rgba(23,35,60,0.15);
}}

.card p {{
    color: {TEXT_SUB};
    font-size: 14.5px;
    line-height: 1.75;
}}

.card b {{
    color: {TEXT};
}}

.card h3 {{
    font-family: 'Space Grotesk', sans-serif;
    color: {TEAL};
    font-size: 15px;
    font-weight: 600;
    margin-top: 20px;
    margin-bottom: 6px;
    text-transform: none;
}}

.card h3:first-child {{
    margin-top: 0;
}}

/* Glow metric cards */

.metric-card {{
    position: relative;
    background: linear-gradient(180deg, {PANEL_ALT}, {PANEL});
    padding: 22px 20px;
    border-radius: 16px;
    border: 1px solid {BORDER};
    text-align: left;
    overflow: hidden;
    box-shadow: 0 10px 28px -14px rgba(23,35,60,0.18);
    transition: border-color 0.25s ease, transform 0.25s ease, box-shadow 0.25s ease;
}}

.metric-card::before {{
    content: "";
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 2px;
    background: linear-gradient(90deg, {TEAL}, {INDIGO});
    box-shadow: 0 0 14px 1px {TEAL};
}}

.metric-card:hover {{
    border-color: {BORDER_STRONG};
    transform: translateY(-2px);
    box-shadow: 0 14px 34px -12px rgba(15,165,150,0.28);
}}

.metric-title {{
    color: {TEXT_MUTED};
    font-size: 12.5px;
    margin-bottom: 10px;
    font-weight: 500;
}}

.metric-value {{
    font-family: 'Space Grotesk', sans-serif;
    color: {TEXT};
    font-size: 30px;
    font-weight: 700;
}}

/* ---------- Workflow timeline ---------- */

.workflow-box {{
    background: {PANEL};
    border: 1px solid {BORDER};
    border-radius: 16px;
    padding: 24px 26px;
    margin-bottom: 20px;
}}

.workflow-title {{
    font-family: 'Space Grotesk', sans-serif;
    font-size: 17px;
    font-weight: 600;
    color: {TEXT};
    margin-bottom: 16px;
}}

.workflow-step {{
    position: relative;
    background: transparent;
    border-left: 2px solid {BORDER_STRONG};
    padding: 8px 0 8px 20px;
    margin: 0;
    font-size: 14px;
    color: {TEXT_SUB};
}}

.workflow-step::before {{
    content: "";
    position: absolute;
    left: -5px;
    top: 15px;
    width: 8px;
    height: 8px;
    border-radius: 50%;
    background: {TEAL};
    box-shadow: 0 0 6px 1px {TEAL};
}}

.workflow-step b {{
    color: {TEXT};
}}

.workflow-desc {{
    color: {TEXT_MUTED};
    font-size: 12.5px;
    line-height: 1.6;
    margin-top: 4px;
    max-width: 640px;
}}

.workflow-step.plain {{
    border-left: 2px solid transparent;
    padding-left: 20px;
    color: {TEXT_MUTED};
    font-size: 13px;
}}

.workflow-step.plain::before {{
    display: none;
}}

/* ---------- Callouts ---------- */

.callout {{
    background: {PANEL};
    border: 1px solid {BORDER};
    border-left: 3px solid var(--callout-color, {TEAL});
    border-radius: 14px;
    padding: 20px 24px;
    margin-bottom: 20px;
    box-shadow: 0 8px 26px -10px rgba(15,165,150,0.35);
    box-shadow: 0 8px 26px -10px color-mix(in srgb, var(--callout-color, {TEAL}) 40%, transparent);
}}

.callout-label {{
    color: {TEXT_MUTED};
    font-size: 12.5px;
    font-weight: 500;
    margin-bottom: 6px;
}}

.callout-value {{
    font-family: 'Space Grotesk', sans-serif;
    color: {TEXT};
    font-size: 22px;
    font-weight: 700;
    margin-bottom: 6px;
}}

.callout-detail {{
    color: {TEXT_SUB};
    font-size: 13.5px;
    line-height: 1.6;
}}

/* ---------- Findings list ---------- */

.finding {{
    display: flex;
    gap: 12px;
    padding: 10px 0;
    border-bottom: 1px solid {BORDER};
    font-size: 14px;
    color: {TEXT_SUB};
    line-height: 1.6;
}}

.finding:last-child {{
    border-bottom: none;
}}

.finding-dot {{
    flex-shrink: 0;
    width: 6px;
    height: 6px;
    border-radius: 50%;
    background: {TEAL};
    box-shadow: 0 0 6px 1px {TEAL};
    margin-top: 8px;
}}

.finding b {{
    color: {TEXT};
}}

/* ---------- Tables ---------- */

[data-testid="stDataFrame"] {{
    border: 1px solid {BORDER};
    border-radius: 12px;
    overflow: hidden;
}}

/* ---------- Footer ---------- */

.footer {{
    text-align: center;
    color: {TEXT_MUTED};
    padding: 40px 0 20px 0;
    font-size: 12.5px;
    border-top: 1px solid {BORDER};
    margin-top: 30px;
}}

</style>
""", unsafe_allow_html=True)


# =========================
# HELPERS
# =========================

def section(title):
    st.markdown(f"""
    <div class="section-header">
        <div class="section-bar"></div>
        <div class="section-title">{title}</div>
    </div>
    """, unsafe_allow_html=True)


def metric(col, label, value):
    with col:
        st.markdown(f"""
        <div class="metric-card">
        <div class="metric-title">{label}</div>
        <div class="metric-value">{value}</div>
        </div>
        """, unsafe_allow_html=True)


def style_fig(fig, height=420):
    fig.update_layout(
        plot_bgcolor=PANEL,
        paper_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Inter, sans-serif", color=TEXT_SUB, size=13),
        title_font=dict(family="Space Grotesk, sans-serif", color=TEXT, size=17),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color=TEXT_SUB)),
        margin=dict(t=60, l=10, r=10, b=10),
        height=height,
    )
    fig.update_xaxes(showgrid=False, linecolor=BORDER, tickfont=dict(color=TEXT_SUB))
    fig.update_yaxes(showgrid=True, gridcolor="rgba(23,35,60,0.07)", zerolinecolor=BORDER, tickfont=dict(color=TEXT_SUB))
    return fig


def callout(label, value, detail, color=TEAL):
    st.markdown(f"""
    <div class="callout" style="--callout-color: {color};">
        <div class="callout-label">{label}</div>
        <div class="callout-value">{value}</div>
        <div class="callout-detail">{detail}</div>
    </div>
    """, unsafe_allow_html=True)


# =========================
# DATA
# =========================

dataset = pd.DataFrame({
    "Class": [
        "Glioma",
        "Meningioma",
        "No Tumor",
        "Pituitary"
    ],
    "Images": [
        1147,
        1329,
        1067,
        1457
    ]
})

model_results = pd.DataFrame({
    "Model": [
        "KNN",
        "Random Forest",
        "Logistic Regression",
        "Decision Tree",
        "EfficientNetB7"
    ],
    "Accuracy": [
        89.10,
        88.70,
        90.90,
        66.70,
        82.20
    ],
    "Macro Precision": [
        89.95,
        89.88,
        91.18,
        68.70,
        83.26
    ],
    "Macro Recall": [
        90.40,
        89.74,
        91.82,
        68.00,
        84.38
    ],
    "Macro F1": [
        89.86,
        89.65,
        91.35,
        67.14,
        83.15
    ]
})


# =========================
# SIDEBAR
# =========================

st.sidebar.markdown("""
<div class="brand-mark">
    <div class="brand-dot"></div>
    <div class="brand-name">BRISC 2025</div>
</div>
""", unsafe_allow_html=True)

st.sidebar.markdown(
    '<div class="brand-sub">Brain Tumor Classification<br>Research Dashboard</div>',
    unsafe_allow_html=True
)

st.sidebar.markdown("---")

page = st.sidebar.radio(
    "EXPLORE PROJECT",
    [
        "Overview",
        "Dataset",
        "Model Performance",
        "EfficientNetB7",
        "Methodology",
        "Project Information"
    ],
    label_visibility="visible"
)

st.sidebar.markdown("---")
st.sidebar.caption("BRISC 2025 • ML Research Project")


# =========================
# OVERVIEW
# =========================

if page == "Overview":

    st.markdown(
        '<div class="main-title">BRISC 2025 Brain Tumor Classification</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="subtitle">'
        'A research dashboard for comparative brain MRI classification using '
        'traditional machine learning and deep learning.'
        '</div>',
        unsafe_allow_html=True
    )

    c1, c2, c3, c4 = st.columns(4)
    metric(c1, "Tumor Classes", "4")
    metric(c2, "Training Images", "5,000")
    metric(c3, "ML Models", "4")
    metric(c4, "Deep Model", "B7")

    section("Project overview")

    st.markdown(f"""
    <div class="card">
    <p>
    The BRISC 2025 project investigates automated brain tumor classification
    from MRI images. The study compares four traditional machine learning
    classifiers with an EfficientNetB7 deep learning approach, evaluated on
    a shared 5,000-image dataset so the comparison is apples-to-apples.
    </p>

    <p>
    The classification task contains four categories:
    <b>Glioma, Meningioma, No Tumor, and Pituitary.</b> Each class is
    represented by roughly a fifth to a third of the dataset, keeping class
    imbalance mild rather than severe.
    </p>

    <p>
    The traditional pipeline uses handcrafted HOG features followed by PCA
    dimensionality reduction, feeding a compact, interpretable feature set
    into four classical classifiers. The deep learning pipeline instead uses
    an ImageNet-pretrained EfficientNetB7 as a transfer-learning feature
    extractor, trading interpretability for the capacity to learn richer
    visual patterns directly from pixels.
    </p>

    <p>
    From a research perspective, the project is designed to examine whether
    a lightweight, handcrafted-feature-based pipeline can remain competitive
    with a modern pretrained deep learning architecture on the same MRI
    classification task. This makes the study useful for understanding the
    trade-off between computational simplicity, feature interpretability,
    and learned visual representation.
    </p>

    <p>
    Both pipelines are scored on the same four metrics — accuracy, macro
    precision, macro recall, and macro F1 — so results on the Model
    Performance page can be read side by side without any unit conversion
    or re-normalization.
    </p>
    </div>
    """, unsafe_allow_html=True)

    section("Model comparison")

    fig = px.bar(
        model_results,
        x="Model",
        y="Accuracy",
        text="Accuracy",
        title="Classification accuracy comparison",
        color="Model",
        color_discrete_sequence=MODEL_COLOR_SEQUENCE,
    )

    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside", marker_line_width=0)
    fig.update_layout(yaxis_title="Accuracy (%)", xaxis_title="", yaxis_range=[0, 100], showlegend=False)
    style_fig(fig)

    st.plotly_chart(fig, width='stretch')

    section("Key findings")

    callout(
        "Best performing model",
        "Logistic Regression — 90.90% accuracy",
        "Highest accuracy and macro F1 (91.35%) among all five evaluated models, "
        "despite using handcrafted HOG features rather than learned representations.",
        color=TEAL,
    )

    st.markdown(f"""
    <div class="card">
        <div class="finding">
            <div class="finding-dot"></div>
            <div>All three of the top traditional classifiers — <b>Logistic Regression,
            KNN, and Random Forest</b> — outperformed the EfficientNetB7 deep learning
            baseline on this dataset.</div>
        </div>
        <div class="finding">
            <div class="finding-dot"></div>
            <div><b>Decision Tree</b> trailed the field by a wide margin (66.70% accuracy),
            consistent with a single tree's tendency to overfit compared to
            ensembled or regularized alternatives.</div>
        </div>
        <div class="finding">
            <div class="finding-dot"></div>
            <div>The HOG + PCA pipeline compressed <b>8,100 raw features</b> down to
            the components explaining 95% of variance before classification.</div>
        </div>
        <div class="finding">
            <div class="finding-dot"></div>
            <div>EfficientNetB7 still reached a respectable <b>82.20% accuracy</b> using
            only ImageNet-pretrained weights, with no dataset-specific fine-tuning
            described in the pipeline.</div>
        </div>
    </div>
    """, unsafe_allow_html=True)


# =========================
# DATASET
# =========================

elif page == "Dataset":

    st.markdown(
        '<div class="main-title">Dataset</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="subtitle">BRISC 2025 training dataset distribution</div>',
        unsafe_allow_html=True
    )

    c1, c2 = st.columns(2)
    metric(c1, "Total Training Images", "5,000")
    metric(c2, "Number of Classes", "4")

    section("Training image distribution")

    fig = px.bar(
        dataset,
        x="Class",
        y="Images",
        text="Images",
        title="BRISC dataset image distribution",
        color="Class",
        color_discrete_map=CLASS_COLORS,
    )

    fig.update_traces(textposition="outside", marker_line_width=0)
    fig.update_layout(xaxis_title="", yaxis_title="Number of images", showlegend=False)
    style_fig(fig)

    st.plotly_chart(fig, width='stretch')

    section("Class share")

    col1, col2 = st.columns([1.1, 1])

    with col1:
        donut = px.pie(
            dataset,
            names="Class",
            values="Images",
            hole=0.58,
            color="Class",
            color_discrete_map=CLASS_COLORS,
        )
        donut.update_traces(textinfo="percent+label", marker=dict(line=dict(color=PANEL, width=2)))
        style_fig(donut, height=380)
        donut.update_layout(showlegend=False)
        st.plotly_chart(donut, width='stretch')

    with col2:
        share_df = dataset.copy()
        share_df["Share"] = (share_df["Images"] / share_df["Images"].sum() * 100).map(lambda x: f"{x:.1f}%")
        st.dataframe(
            share_df,
            width='stretch',
            hide_index=True
        )
        st.markdown(f"""
        <div class="card" style="margin-top: 4px;">
        <p>
        Class sizes are fairly balanced, with the smallest class (No Tumor,
        21.3%) and largest (Pituitary, 29.1%) differing by roughly 390 images.
        This limits the risk of a model learning to favor one class purely
        from imbalance.
        </p>
        </div>
        """, unsafe_allow_html=True)


# =========================
# MODEL PERFORMANCE
# =========================

elif page == "Model Performance":

    st.markdown(
        '<div class="main-title">Model Performance</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="subtitle">'
        'Comparative performance of the evaluated classification models'
        '</div>',
        unsafe_allow_html=True
    )

    callout(
        "Top result",
        "Logistic Regression — 90.90% accuracy",
        "The strongest macro F1 score (91.35%) also belongs to Logistic "
        "Regression, indicating balanced performance across all four classes "
        "rather than strength on just one.",
        color=TEAL,
    )

    section("Performance summary")

    display_df = model_results.copy()

    display_df["Accuracy"] = display_df["Accuracy"].map(lambda x: f"{x:.2f}%")
    display_df["Macro Precision"] = display_df["Macro Precision"].map(lambda x: f"{x:.2f}%")
    display_df["Macro Recall"] = display_df["Macro Recall"].map(lambda x: f"{x:.2f}%")
    display_df["Macro F1"] = display_df["Macro F1"].map(lambda x: f"{x:.2f}%")

    st.dataframe(
        display_df,
        width='stretch',
        hide_index=True
    )

    section("Accuracy comparison")

    fig = px.bar(
        model_results,
        x="Model",
        y="Accuracy",
        text="Accuracy",
        color="Model",
        color_discrete_sequence=MODEL_COLOR_SEQUENCE,
    )

    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside", marker_line_width=0)
    fig.update_layout(yaxis_title="Accuracy (%)", xaxis_title="", yaxis_range=[0, 100], showlegend=False)
    style_fig(fig)

    st.plotly_chart(fig, width='stretch')

    section("Macro metrics")

    metrics_df = model_results.melt(
        id_vars="Model",
        value_vars=[
            "Macro Precision",
            "Macro Recall",
            "Macro F1"
        ],
        var_name="Metric",
        value_name="Score"
    )

    fig2 = px.bar(
        metrics_df,
        x="Model",
        y="Score",
        color="Metric",
        barmode="group",
        color_discrete_sequence=[TEAL, INDIGO, AMBER],
    )

    fig2.update_traces(marker_line_width=0)
    fig2.update_layout(yaxis_title="Score (%)", xaxis_title="", yaxis_range=[0, 100])
    style_fig(fig2)

    st.plotly_chart(fig2, width='stretch')

    section("Metrics explained")

    st.markdown("""
    <div class="card">

    <h3>Accuracy</h3>
    <p>The share of all predictions the model got right. Simple to read, but can be
    misleading if classes are imbalanced.</p>

    <h3>Macro precision</h3>
    <p>Of everything the model labeled as a given class, the share that was actually
    that class — averaged evenly across all four classes.</p>

    <h3>Macro recall</h3>
    <p>Of everything that truly belongs to a given class, the share the model
    correctly found — averaged evenly across all four classes.</p>

    <h3>Macro F1</h3>
    <p>The harmonic mean of macro precision and macro recall, giving a single
    balanced score that penalizes models which trade one off heavily for the other.</p>

    </div>
    """, unsafe_allow_html=True)


# =========================
# EFFICIENTNET
# =========================

elif page == "EfficientNetB7":

    st.markdown(
        '<div class="main-title">EfficientNetB7</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="subtitle">'
        'Deep learning approach for four-class brain MRI classification'
        '</div>',
        unsafe_allow_html=True
    )

    c1, c2, c3, c4 = st.columns(4)
    metric(c1, "Architecture", "B7")
    metric(c2, "Input Size", "600×600")
    metric(c3, "Dropout", "0.3")
    metric(c4, "Output Classes", "4")

    section("Architecture")

    st.markdown(f"""
    <div class="card">
    <p>
    EfficientNetB7 uses <b>ImageNet pretrained weights</b> as the feature
    extraction backbone.
    </p>

    <p>
    The architecture processes 600×600 RGB MRI images, followed by Global
    Average Pooling, Dropout (0.3), and a Dense Softmax layer for four-class
    classification.
    </p>
    </div>
    """, unsafe_allow_html=True)

    section("Reported performance")

    b7_metrics = pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Macro Precision",
            "Macro Recall",
            "Macro F1"
        ],
        "Score": [
            "82.20%",
            "83.26%",
            "84.38%",
            "83.15%"
        ]
    })

    st.dataframe(
        b7_metrics,
        width='stretch',
        hide_index=True
    )


# =========================
# METHODOLOGY
# =========================

elif page == "Methodology":

    st.markdown(
        '<div class="main-title">Methodology</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="subtitle">'
        'Research workflow used for traditional ML and deep learning'
        '</div>',
        unsafe_allow_html=True
    )

    c1, c2, c3, c4 = st.columns(4)
    metric(c1, "Pipelines Compared", "2")
    metric(c2, "Raw HOG Features", "8,100")
    metric(c3, "PCA Variance Kept", "95%")
    metric(c4, "Classical Models", "4")

    section("A. Traditional machine learning pipeline")

    st.markdown(f"""
    <div class="card">
    <p>
    The traditional machine learning pipeline follows a structured
    feature-engineering approach. Instead of allowing a neural network to
    learn directly from raw MRI pixels, visual information is first converted
    into handcrafted HOG features. PCA is then applied to reduce the
    dimensionality of the feature space before the data is passed to four
    different classifiers.
    </p>

    <p>
    This pipeline is useful for comparison because it requires substantially
    less computational complexity than a deep neural network and allows the
    effect of different classical classifiers to be studied using the same
    feature representation.
    </p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown(f"""
    <div class="workflow-box">

    <div class="workflow-step"><b>1. MRI images</b>
    <div class="workflow-desc">Raw brain MRI scans across the four target classes form the
    starting point for the classical pipeline.</div></div>

    <div class="workflow-step"><b>2. Grayscale conversion</b>
    <div class="workflow-desc">Color channels are collapsed to a single intensity channel,
    since HOG operates on gradient direction and magnitude rather than color.</div></div>

    <div class="workflow-step"><b>3. Resize (128×128)</b>
    <div class="workflow-desc">Every image is standardized to a fixed resolution so the
    resulting feature vectors are the same length regardless of source image size.</div></div>

    <div class="workflow-step"><b>4. Normalization [0,1]</b>
    <div class="workflow-desc">Pixel intensities are rescaled to a common range, which keeps
    gradient magnitudes comparable across images taken under different scan conditions.</div></div>

    <div class="workflow-step"><b>5. HOG feature extraction (8,100 features)</b>
    <div class="workflow-desc">Histogram of Oriented Gradients captures local edge and shape
    structure by summarizing gradient orientations in small image cells — well suited to the
    textured, structural patterns of tumor tissue.</div></div>

    <div class="workflow-step"><b>6. PCA (95% variance)</b>
    <div class="workflow-desc">Principal Component Analysis compresses the 8,100-dimensional
    HOG vector down to the smaller set of components that together explain 95% of the
    variance, reducing noise and training time.</div></div>

    <div class="workflow-step"><b>7. ML classifiers</b>
    <div class="workflow-desc">The reduced feature set is passed to four classical
    algorithms, trained and evaluated independently.</div></div>
    <div class="workflow-step plain">KNN • Random Forest • Logistic Regression • Decision Tree</div>

    <div class="workflow-step"><b>8. Prediction</b>
    <div class="workflow-desc">Each classifier outputs one of the four tumor categories for
    a given image.</div></div>

    </div>
    """, unsafe_allow_html=True)

    section("B. Deep learning pipeline")

    st.markdown(f"""
    <div class="card">
    <p>
    The deep learning pipeline follows a transfer-learning strategy using
    EfficientNetB7. Instead of manually designing image features, the network
    uses representations learned from large-scale ImageNet training and
    applies them to the brain MRI classification task.
    </p>

    <p>
    The MRI images are resized to 600 × 600 RGB inputs before being processed
    by the EfficientNetB7 backbone. The extracted visual representations are
    then passed through Global Average Pooling, Dropout, and a four-class
    Softmax classification layer.
    </p>

    <p>
    This approach allows the model to capture more complex visual patterns
    than handcrafted features while still benefiting from pretrained
    representations.
    </p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown(f"""
    <div class="workflow-box">

    <div class="workflow-step"><b>1. MRI images</b>
    <div class="workflow-desc">The same underlying MRI scans, fed through a separate
    pipeline built around a pretrained convolutional network.</div></div>

    <div class="workflow-step"><b>2. Resize & normalize (600×600 RGB)</b>
    <div class="workflow-desc">Images are kept in full color and resized to
    EfficientNetB7's expected 600×600 input resolution.</div></div>

    <div class="workflow-step"><b>3. EfficientNetB7 (ImageNet pretrained weights)</b>
    <div class="workflow-desc">The backbone was pretrained on ImageNet and reused here as a
    general-purpose visual feature extractor via transfer learning, rather than trained
    from scratch.</div></div>

    <div class="workflow-step"><b>4. Global average pooling</b>
    <div class="workflow-desc">Collapses each feature map produced by the backbone into a
    single value per channel, turning a spatial feature grid into one compact vector.</div></div>

    <div class="workflow-step"><b>5. Dropout (0.3)</b>
    <div class="workflow-desc">Randomly disables 30% of the pooled features during training
    to discourage over-reliance on any single feature and reduce overfitting.</div></div>

    <div class="workflow-step"><b>6. Dense softmax (4 classes)</b>
    <div class="workflow-desc">A final fully-connected layer converts the pooled features
    into a probability distribution across the four tumor classes.</div></div>

    <div class="workflow-step"><b>7. Prediction</b>
    <div class="workflow-desc">The class with the highest softmax probability is taken as
    the model's prediction.</div></div>

    </div>
    """, unsafe_allow_html=True)

    section("Feature engineering, in plain terms")

    st.markdown("""
    <div class="card">

    <h3>Histogram of Oriented Gradients (HOG)</h3>
    <p>
    HOG describes an image by the direction its edges point in, rather than by raw pixel
    values. It divides the image into small cells, builds a histogram of gradient
    directions in each cell, and concatenates those histograms into one long feature
    vector. This makes it sensitive to shape and structure while staying fairly robust to
    lighting differences between scans.
    </p>

    <h3>Principal Component Analysis (PCA)</h3>
    <p>
    PCA looks for the directions in the 8,100-dimensional HOG feature space that carry the
    most variation between images, and re-expresses each image using only those directions.
    Keeping enough components to explain 95% of the variance discards mostly redundant or
    noisy information while retaining the signal that actually separates tumor classes.
    </p>

    </div>
    """, unsafe_allow_html=True)

    section("Classifier algorithms compared")

    st.markdown("""
    <div class="card">

    <h3>K-Nearest Neighbors (KNN)</h3>
    <p>Classifies an image by looking at the closest training examples in feature space and
    taking a majority vote of their labels. Simple and non-parametric, but sensitive to the
    scale and quality of the feature space it's given.</p>

    <h3>Random Forest</h3>
    <p>An ensemble of many decision trees, each trained on a random subset of data and
    features, with predictions combined by voting. Averaging across trees typically reduces
    the overfitting that a single tree is prone to.</p>

    <h3>Logistic Regression</h3>
    <p>Fits a linear decision boundary between classes by optimizing class probabilities
    directly. Despite its simplicity, it performs well when the PCA-reduced feature space is
    close to linearly separable.</p>

    <h3>Decision Tree</h3>
    <p>Splits the feature space into regions using a sequence of threshold rules learned
    from the data. A single tree is easy to interpret but tends to overfit more than
    ensembled alternatives like Random Forest.</p>

    </div>
    """, unsafe_allow_html=True)

    section("Why compare classical ML and deep learning")

    st.markdown("""
    <div class="card">

    <p>
    The methodology evaluates two fundamentally different approaches to brain tumor
    classification side by side, rather than assuming one is categorically better.
    </p>

    <p>
    The first approach extracts handcrafted HOG features from preprocessed MRI images and
    reduces their dimensionality using PCA before classification. It is computationally
    lightweight, fast to train, and its features are easier to reason about — a histogram
    of edge directions is a concrete, inspectable quantity.
    </p>

    <p>
    The second approach uses transfer learning with EfficientNetB7 and ImageNet pretrained
    weights to learn image representations directly, trading interpretability and training
    simplicity for the capacity to learn more complex visual patterns.
    </p>

    <p>
    Running both side by side on the same dataset and the same evaluation metrics is what
    makes the comparison on the Model Performance page meaningful, rather than comparing
    numbers from unrelated studies.
    </p>

    </div>
    """, unsafe_allow_html=True)

    section("End-to-end research workflow")

    st.markdown(f"""
    <div class="workflow-box">

    <div class="workflow-step">
    <b>1. Dataset preparation</b>
    <div class="workflow-desc">
    MRI images are organized according to the four target classes:
    Glioma, Meningioma, No Tumor, and Pituitary.
    </div>
    </div>

    <div class="workflow-step">
    <b>2. Image preprocessing</b>
    <div class="workflow-desc">
    Images are standardized according to the requirements of each
    classification pipeline.
    </div>
    </div>

    <div class="workflow-step">
    <b>3. Feature representation</b>
    <div class="workflow-desc">
    The traditional pipeline extracts HOG features, while EfficientNetB7
    learns visual representations through its pretrained convolutional
    backbone.
    </div>
    </div>

    <div class="workflow-step">
    <b>4. Dimensionality reduction</b>
    <div class="workflow-desc">
    PCA is applied to the HOG representation while retaining 95% of the
    variance before classical model training.
    </div>
    </div>

    <div class="workflow-step">
    <b>5. Model training</b>
    <div class="workflow-desc">
    KNN, Random Forest, Logistic Regression, Decision Tree, and EfficientNetB7
    are trained using their respective pipelines.
    </div>
    </div>

    <div class="workflow-step">
    <b>6. Classification</b>
    <div class="workflow-desc">
    Each model predicts one of the four brain MRI categories.
    </div>
    </div>

    <div class="workflow-step">
    <b>7. Performance comparison</b>
    <div class="workflow-desc">
    Models are compared using accuracy, macro precision, macro recall, and
    macro F1-score to provide a consistent evaluation framework.
    </div>
    </div>

    </div>
    """, unsafe_allow_html=True)


# =========================
# PROJECT INFORMATION
# =========================

elif page == "Project Information":

    st.markdown(
        '<div class="main-title">Project Information</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        '<div class="subtitle">BRISC 2025 research project details</div>',
        unsafe_allow_html=True
    )

    c1, c2, c3, c4 = st.columns(4)
    metric(c1, "Tumor Classes", "4")
    metric(c2, "Training Images", "5,000")
    metric(c3, "Classical Models", "4")
    metric(c4, "Deep Model", "EfficientNetB7")

    section("Project summary")

    st.markdown("""
    <div class="card">

    <h3>Project title</h3>
    <p><b>Brain Tumor Classification Using Machine Learning and Deep Learning</b></p>

    <h3>Dataset</h3>
    <p>BRISC 2025 Brain MRI Dataset — 5,000 labeled MRI images across four classes.</p>

    <h3>Classification classes</h3>
    <p>Glioma • Meningioma • No Tumor • Pituitary</p>

    <h3>Traditional machine learning</h3>
    <p>KNN, Random Forest, Logistic Regression, and Decision Tree, trained on
    HOG features reduced with PCA.</p>

    <h3>Deep learning</h3>
    <p>EfficientNetB7 with ImageNet pretrained weights, used as a transfer-learning
    feature extractor.</p>

    <h3>Project focus</h3>
    <p>
    Comparative evaluation of handcrafted-feature-based machine learning
    and transfer-learning-based deep learning approaches for brain MRI
    classification, judged on accuracy and macro-averaged precision, recall,
    and F1.
    </p>

    </div>
    """, unsafe_allow_html=True)

    section("Model roster")

    roster = pd.DataFrame({
        "Model": ["Logistic Regression", "KNN", "Random Forest", "EfficientNetB7", "Decision Tree"],
        "Type": ["Classical (linear)", "Classical (instance-based)", "Classical (ensemble)", "Deep learning (transfer learning)", "Classical (single tree)"],
        "Accuracy": ["90.90%", "89.10%", "88.70%", "82.20%", "66.70%"],
    })

    st.dataframe(roster, width='stretch', hide_index=True)

    section("Limitations & future work")

    st.markdown("""
    <div class="card">
        <div class="finding">
            <div class="finding-dot"></div>
            <div>Class sizes are close but not identical (21.3%–29.1% share), so
            a small amount of class imbalance remains a factor worth controlling
            for in future evaluation.</div>
        </div>
        <div class="finding">
            <div class="finding-dot"></div>
            <div>The results shown reflect a single evaluation run per model;
            reporting results across multiple cross-validation folds would give
            a clearer picture of variance in each score.</div>
        </div>
        <div class="finding">
            <div class="finding-dot"></div>
            <div>EfficientNetB7 was used as a pretrained feature extractor.
            Fine-tuning the backbone end-to-end on brain MRI data specifically,
            rather than relying solely on ImageNet features, is a natural next
            step to explore.</div>
        </div>
        <div class="finding">
            <div class="finding-dot"></div>
            <div>Data augmentation (rotation, flipping, contrast jitter) was not
            described as part of either pipeline and could help both approaches
            generalize beyond this specific dataset.</div>
        </div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown(
        '<div class="footer">BRISC 2025 • Brain Tumor Classification Research Dashboard</div>',
        unsafe_allow_html=True
    )

Writing /content/app.py


In [19]:
!find /content/drive/MyDrive -name "app.py"

In [16]:
!cat /content/streamlit.log

cat: /content/streamlit.log: No such file or directory


In [17]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [5]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [6]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [8]:
!streamlit run app.py

/bin/bash: line 1: streamlit: command not found


In [9]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 74.8 MB/s eta 0:00:00


In [23]:
!streamlit run app.py



2026-09-14 06:43:28.132 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.75.121.189:8501

2026-09-14 06:43:43.632 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-use1b1-zf4562w1jp5r-b.us-east1-1.prod.colab.dev, host=m-s-kkb-use1b1-zf4562w1jp5r.us-east1-b.c.codatalab-user-runtimes.internal:8007
2026-09-14 06:43:47.061 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-use1b1-zf4562w1jp5r-b.us-east1-1.prod.colab.dev, host=m-s-kkb-use1b1-zf4562w1jp5r.us-east1-b.c.codatalab-user-runtimes.internal:8007
2026-09-14 06:43:51.265 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-use1b1-zf4562w1jp5r-b.us-east1-1.prod.colab.dev, host=m-s-kkb-use1b1-zf4562w1jp5r.us-east1-b.c.codatalab-user-runtimes.internal:8007
2

In [22]:
!pkill -f streamlit

In [21]:
!curl -I http://localhost:8501

curl: (7) Failed to connect to localhost port 8501 after 0 ms: Couldn't connect to server


In [24]:
!streamlit run /content/app.py \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  --browser.gatherUsageStats false \
  > /content/streamlit.log 2>&1 &

In [40]:
!cat /content/streamlit.log

2026-09-14 04:50:47.898 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.138.193.210:8501



In [ ]:
!mkdir -p /content/.streamlit

In [ ]:
import os
os.makedirs("/content/.streamlit", exist_ok=True)
with open("/content/.streamlit/config.toml", "w") as f:
    f.write('''[theme]
base = "light"
primaryColor = "#0FA596"
backgroundColor = "#F5F8FC"
secondaryBackgroundColor = "#FFFFFF"
textColor = "#17233C"
font = "sans serif"
''')

In [ ]:
!pkill -f streamlit

In [ ]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8501)")
print(url)

https://8501-m-s-kkb-use1c2-3qalfvvmnhz2z-c.us-east1-2.prod.colab.dev


In [ ]:
%%writefile /content/requirements.txt
streamlit
pandas
plotly

Writing /content/requirements.txt


In [ ]:
import os

print("app.py:", os.path.exists("/content/app.py"))
print("requirements.txt:", os.path.exists("/content/requirements.txt"))

app.py: True
requirements.txt: True


In [ ]:
from google.colab import files
files.download("/content/app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("/content/requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>